# Lección 7: Programación Lineal y Optimización Matemática
### Álgebra Lineal Computacional y Fundamentos de la Física Matemática

---

Este cuaderno de Jupyter Colab establece los fundamentos algebraicos, geométricos y algorítmicos de la **programación lineal**. La programación (o planificación) lineal es una técnica matemática para maximizar o minimizar una función objetivo de primer grado sujeta a restricciones de igualdad y desigualdad lineales. Estos modelos son fundamentales para la optimización de recursos en la industria, logística, planificación financiera y modelado energético en física aplicada. El tratamiento combina el rigor analítico formal con implementaciones interactivas en Python, visualizaciones de regiones factibles acotadas y no acotadas, y la resolución exacta mediante la librería científica `scipy.optimize.linprog`.

---


## Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Construir** modelos matemáticos de optimización lineal a partir de descripciones verbales de problemas.
2. **Resolver** problemas de programación lineal de dos variables utilizando el método gráfico, identificando la región factible y los vértices (puntos esquina).
3. **Transformar** modelos de programación lineal a su forma estándar y a la forma de holgura (distensionada), introduciendo variables de holgura y exceso.
4. **Comprender** la teoría de la dualidad, el Teorema Débil de la Dualidad y el Teorema Fuerte de la Dualidad.
5. **Formular** el problema dual asociado a un problema primal y viceversa.
6. **Utilizar** la biblioteca científica de Python `scipy.optimize.linprog` para resolver problemas primales y duales de programación lineal de forma automatizada y verificar sus resultados.


## 1. Conceptos Generales de la Programación Lineal

La **programación lineal** (también llamada optimización lineal) es una rama de la optimización matemática dedicada a resolver problemas donde la meta (el beneficio máximo o el costo mínimo) y los requisitos (restricciones) se modelan mediante relaciones lineales.

### 1.1 Elementos del Modelo
Un modelo estándar está compuesto por:
1. **Variables de decisión:** Las incógnitas del problema, típicamente denotadas por $x_1, x_2, \dots, x_n$.
2. **Función Objetivo:** La expresión lineal de primer orden que se desea maximizar o minimizar.
   $$z = c_1 x_1 + c_2 x_2 + \dots + c_n x_n$$
3. **Restricciones:** Desigualdades o igualdades lineales de primer orden que representan las limitaciones físicas o económicas (recursos limitados).
   $$a_{i1} x_1 + a_{i2} x_2 + \dots + a_{in} x_n \le b_i \quad \text{para } i = 1, \dots, m$$
4. **Condición de no negatividad:** Las variables no pueden tomar valores negativos:
   $$x_j \ge 0 \quad \text{para } j = 1, \dots, n$$

### 1.2 Región Factible y Politopo Convexo
El conjunto de todas las soluciones que cumplen con todas las restricciones a la vez se conoce como **Región Factible ($F$)**:
$$F = \{x \in \mathbb{R}^n \mid A x \le b, \ x \ge 0\}$$
Geométricamente, la región factible define un **politopo convexo** (la intersección de múltiples semiespacios lineales).
* **Teorema Fundamental de la Programación Lineal:** Si un problema de programación lineal tiene una solución óptima y la región factible es acotada, dicha solución óptima se encuentra siempre en al menos uno de los **vértices (puntos esquina)** del politopo factible.

### 1.3 Cronología de la Programación Lineal
* **1947:** George Dantzig publica el **Algoritmo Simplex**, revolucionando el campo al proveer un método combinatorio para recorrer los vértices eficientemente.
* **1947:** John von Neumann desarrolla la **Teoría de la Dualidad** en colaboración con Dantzig. De forma independiente, Leonid Kantórovich formula la teoría aplicada a la planificación económica (por la cual recibió el Premio Nobel de Economía en 1975).
* **1984:** Narendra Karmarkar introduce el **Método del Punto Interior**, que resuelve sistemas de programación lineal en tiempo polinomial atravesando el interior del politopo.


## 2. El Método Gráfico

El método gráfico es una técnica intuitiva para resolver problemas de programación lineal con dos variables de decisión ($x$ e $y$). Consiste en trazar las rectas de restricción, sombrear la región factible, identificar las coordenadas de los vértices y evaluar la función objetivo en ellos para encontrar el valor óptimo.

### 2.1 Caso de Estudio: Fábrica de Bicicletas (Boirivant [1])
Una fábrica produce dos tipos de bicicletas:
* **Bicicletas de montaña ($x$):** Precio de venta de 200 €, requiere 1 kg de acero y 3 kg de aluminio.
* **Bicicletas de paseo ($y$):** Precio de venta de 150 €, requiere 2 kg de acero y 2 kg de aluminio.
* **Disponibilidad:** La fábrica cuenta únicamente con 80 kg de acero y 120 kg de aluminio.

#### Formulación Matemática:
Queremos maximizar el beneficio total:
$$\text{Maximizar } z = 200x + 150y$$
Sujeto a las restricciones:
* Acero: $1x + 2y \le 80$
* Aluminio: $3x + 2y \le 120$
* No negatividad: $x \ge 0, \ y \ge 0$

#### Cálculo de Vértices:
1. **Origen:** $(0, 0)$
2. **Intersección de la restricción de acero con el eje Y ($x=0$):**
   $$0 + 2y = 80 \implies y = 40 \implies (0, 40)$$
3. **Intersección de la restricción de aluminio con el eje X ($y=0$):**
   $$3x + 0 = 120 \implies x = 40 \implies (40, 0)$$
4. **Intersección entre acero y aluminio:**
   $$x + 2y = 80 \implies x = 80 - 2y$$
   Sustituyendo en la otra restricción:
   $$3(80 - 2y) + 2y = 120 \implies 240 - 4y = 120 \implies 4y = 120 \implies y = 30 \implies x = 20 \implies (20, 30)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definimos las funciones de restricción para graficar sus límites
# x + 2y = 80  => y = (80 - x)/2
# 3x + 2y = 120 => y = (120 - 3x)/2
x_vals = np.linspace(0, 50, 400)
y_acero = (80 - x_vals) / 2.0
y_aluminio = (120 - 3 * x_vals) / 2.0

plt.figure(figsize=(10, 8))

# Graficamos las líneas de las restricciones
plt.plot(x_vals, y_acero, 'b-', label='Acero: $x + 2y \\le 80$')
plt.plot(x_vals, y_aluminio, 'r-', label='Aluminio: $3x + 2y \\le 120$')

# Graficamos la región factible (intersección de las restricciones)
# El límite superior de y está determinado por el mínimo entre las dos funciones de restricción
y_factible = np.minimum(y_acero, y_aluminio)
# Omitimos valores negativos de y
y_factible = np.maximum(y_factible, 0)
plt.fill_between(x_vals, 0, y_factible, where=(x_vals <= 40), color='green', alpha=0.3, label='Region Factible (Acotada)')

# Marcamos y etiquetamos los vértices del polígono
vertices = [(0, 0), (40, 0), (0, 40), (20, 30)]
beneficios = [200*v[0] + 150*v[1] for v in vertices]

for v, ben in zip(vertices, beneficios):
    color = 'gold' if v == (20, 30) else 'black'
    plt.scatter(v[0], v[1], color=color, s=120, zorder=5)
    plt.annotate(f"Vertice {v}\nBeneficio: {ben} EUR", xy=v, xytext=(v[0]+1, v[1]+1),
                 fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5))

plt.title('Metodo Grafico - Optimizacion de Produccion de Bicicletas', fontsize=13, fontweight='bold')
plt.xlabel('Bicicletas de Montaña (x)', fontsize=11)
plt.ylabel('Bicicletas de Paseo (y)', fontsize=11)
plt.xlim(-5, 50)
plt.ylim(-5, 50)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.show()



## 3. Formas Alternativas: Estándar y de Holgura

Un problema de programación lineal puede expresarse en diferentes formatos equivalentes.

### 3.1 Forma Estándar
Es el formato canónico para plantear el modelo matemático. Consta de:
* Una función lineal a **maximizar** o **minimizar**.
* Restricciones modeladas exclusivamente como desigualdades del tipo **menor o igual ($\le$)** para maximización, o **mayor o igual ($\ge$)** para minimización.
* Todas las variables de decisión restringidas a valores **no negativos** ($x_j \ge 0$).

#### Transformaciones a la Forma Estándar:
1. **Signo de la variable irrestricta:** Si una variable $x_i$ puede ser negativa, la reemplazamos por la diferencia de dos nuevas variables no negativas:
   $$x_i = x'_i - x''_i \quad \text{con} \quad x'_i \ge 0, \ x''_i \ge 0$$
2. **Tipo de Optimización:** Cambiar maximización por minimización o viceversa multiplicando los coeficientes de coste por $-1$:
   $$\max z = - \min (-z)$$

### 3.2 Forma de Holgura (Slack Form) o Aumentada
Para poder aplicar el Algoritmo Simplex, es necesario transformar las restricciones de desigualdad en igualdades. Esto se logra introduciendo variables auxiliares no negativas:

* **Variable de Holgura (slack variable $s \ge 0$):** Se suma al lado izquierdo de restricciones del tipo "menor o igual ($\le$)" para absorber la diferencia hasta el recurso limitante.
   $$a_1 x_1 + a_2 x_2 \le b \implies a_1 x_1 + a_2 x_2 + s = b \quad (s \ge 0)$$
* **Variable de Exceso (surplus variable $s \ge 0$):** Se resta en restricciones del tipo "mayor o igual ($\ge$)".
   $$a_1 x_1 + a_2 x_2 \ge b \implies a_1 x_1 + a_2 x_2 - s = b \quad (s \ge 0)$$

En forma matricial de bloques, la forma distensionada se unifica como:
$$\begin{pmatrix} 1 & -c^T & 0 \\ 0 & A & I \end{pmatrix} \begin{pmatrix} z \\ X \\ S \end{pmatrix} = \begin{pmatrix} 0 \\ b \end{pmatrix}$$
Donde $S$ es el vector de variables de holgura recientemente introducidas.


In [ ]:
# Función para transformar restricciones de desigualdad a forma de holgura (igualdades)
def convertir_a_holgura(A, b):
    # A es la matriz de coeficientes (m x n), b es el vector RHS (m)
    m, n = A.shape
    # Creamos una matriz identidad de tamaño m x m para las variables de holgura
    I_slack = np.eye(m)
    # Concatenamos la matriz original con la identidad de holgura
    A_slack = np.hstack((A, I_slack))
    return A_slack, b

# Probamos con las restricciones del problema de las bicicletas:
# 1x + 2y <= 80
# 3x + 2y <= 120
A_ineq = np.array([
    [1.0, 2.0],
    [3.0, 2.0]
])
b_ineq = np.array([80.0, 120.0])

A_eq, b_eq = convertir_a_holgura(A_ineq, b_ineq)
print("Matriz A original de restricciones:")
print(A_ineq)
print("\nMatriz A_eq en Forma de Holgura (con variables s1 y s2 añadidas):")
print(A_eq)
print("\nRestricciones en formato de igualdad:")
for i in range(len(b_eq)):
    print(f"  {A_eq[i, 0]}*x + {A_eq[i, 1]}*y + {A_eq[i, 2]}*s1 + {A_eq[i, 3]}*s2 == {b_eq[i]}")



## 4. Teoría de la Dualidad en Programación Lineal

La dualidad es uno de los conceptos más profundos de la optimización matemática. Establece que cada problema de programación lineal (llamado **Primal**) tiene asociado un problema simétrico alternativo (llamado **Dual**).

### 4.1 Definición y Mapeo Primal-Dual

| Aspecto | Problema Primal | Problema Dual |
| :--- | :---: | :---: |
| **Optimización** | Maximizar $z = c^T x$ | Minimizar $w = b^T y$ |
| **Restricciones** | $A x \le b$ | $A^T y \ge c$ |
| **Variables** | $x \ge 0$ | $y \ge 0$ |

### 4.2 Teoremas de Dualidad
* **Teorema Débil de la Dualidad:** Para cualquier solución factible $x$ del primal y cualquier solución factible $y$ del dual, se cumple que el valor objetivo del primal es un límite inferior del dual:
  $$c^T x \le b^T y$$
* **Teorema Fuerte de la Dualidad:** Si uno de los dos problemas alcanza una solución óptima ($x^*$ para el primal e $y^*$ para el dual), entonces el otro también lo hace y los valores óptimos de sus funciones objetivo son **idénticos**:
  $$c^T x^* = b^T y^*$$

### 4.3 Caso de Estudio: Alimentación de Animales (Sotelo [5])

Consideramos las necesidades de vitaminas A, B y C para ganado a partir de dos alimentos, $F_1$ y $F_2$:

| Vitamina | Contenido en $F_1$ (u/g) | Contenido en $F_2$ (u/g) | Requerimiento Mínimo Diario |
| :--- | :---: | :---: | :---: |
| **Vitamina A** | 1 | 2 | 120 unidades |
| **Vitamina B** | 2 | 3 | 180 unidades |
| **Vitamina C** | 3 | 1 | 150 unidades |
| **Costo por gramo** | 10 ¢ | 12 ¢ | - |

#### Aclaración Académica (Corrección de Errata en el PDF original):
El PDF oficial indica en su tabla de evaluación de vértices el punto esquina **$(36, 45)$** con un costo de **$864$ ¢**. Sin embargo, esto contiene un error tipográfico en la coordenada $y_2$:
* Si evaluamos $(36, 45)$, el costo es: $10(36) + 12(45) = 360 + 540 = 900$ ¢.
* Si calculamos algebraicamente la intersección exacta de la frontera de las restricciones de Vitamina A ($y_1 + 2y_2 = 120$) y Vitamina C ($3y_1 + y_2 = 150$), obtenemos:
  $$y_1 = 120 - 2y_2 \implies 3(120 - 2y_2) + y_2 = 150 \implies 360 - 5y_2 = 150 \implies 5y_2 = 210 \implies y_2 = 42$$
  $$y_1 = 120 - 2(42) = 120 - 84 = 36$$
El punto de intersección correcto es **$(36, 42)$**. Evaluando este punto:
$$C = 10(36) + 12(42) = 360 + 504 = 864 \text{ centavos } (8.64 \text{ EUR})$$
En las siguientes celdas demostraremos de forma computacional que el óptimo exacto ocurre en $(36, 42)$ y resolveremos el problema dual para comprobar la coincidencia de valores objetivos.

#### Modelado del Problema Primal (Minimización del Costo):
$$\text{Minimizar } C = 10y_1 + 12y_2$$
Sujeto a:
* Vitamina A: $y_1 + 2y_2 \ge 120$
* Vitamina B: $2y_1 + 3y_2 \ge 180$
* Vitamina C: $3y_1 + y_2 \ge 150$
* No negatividad: $y_1, y_2 \ge 0$

#### Modelado del Problema Dual (Maximización de Ganancias):
$$\text{Maximizar } P = 120x_1 + 180x_2 + 150x_3$$
Sujeto a:
* Alimento 1: $x_1 + 2x_2 + 3x_3 \le 10$
* Alimento 2: $2x_1 + 3x_2 + x_3 \le 12$
* No negatividad: $x_1, x_2, x_3 \ge 0$


In [ ]:
from scipy.optimize import linprog

# 1. RESOLUCIÓN DEL PROBLEMA PRIMAL (Minimización)
# scipy.optimize.linprog por defecto resuelve minimizaciones: min c^T x s.a. A_ub x <= b_ub
# Coeficientes de la función objetivo: C = 10*y1 + 12*y2
c_primal = [10.0, 12.0]

# Restricciones de desigualdad mayor o igual (>=) se convierten a (<=) multiplicando por -1
A_primal = [
    [-1.0, -2.0],  # -y1 - 2y2 <= -120
    [-2.0, -3.0],  # -2y1 - 3y2 <= -180
    [-3.0, -1.0]   # -3y1 - y2 <= -150
]
b_primal = [-120.0, -180.0, -150.0]

# Límites de las variables (y1 >= 0, y2 >= 0)
limites_y = [(0, None), (0, None)]

res_primal = linprog(c_primal, A_ub=A_primal, b_ub=b_primal, bounds=limites_y, method='highs')

print("--- SOLUCION DEL PROBLEMA PRIMAL ---")
print(f"Estado de la solucion: {res_primal.message}")
print(f"Costo minimo optimo: {res_primal.fun:.4f} centavos (esperado: 864.0)")
print(f"Cantidades optimas a comprar:")
print(f"  y1 (Alimento 1) = {res_primal.x[0]:.4f} g (esperado: 36.0)")
print(f"  y2 (Alimento 2) = {res_primal.x[1]:.4f} g (esperado: 42.0)")
print("-" * 50)


# 2. RESOLUCIÓN DEL PROBLEMA DUAL (Maximización)
# Para maximizar P = 120x1 + 180x2 + 150x3, minimizamos -P
c_dual = [-120.0, -180.0, -150.0]

# Restricciones de desigualdad (<=)
A_dual = [
    [1.0, 2.0, 3.0],  # x1 + 2x2 + 3x3 <= 10
    [2.0, 3.0, 1.0]   # 2x1 + 3x2 + x3 <= 12
]
b_dual = [10.0, 12.0]

limites_x = [(0, None), (0, None), (0, None)]

res_dual = linprog(c_dual, A_ub=A_dual, b_ub=b_dual, bounds=limites_x, method='highs')

print("--- SOLUCION DEL PROBLEMA DUAL ---")
print(f"Estado de la solucion: {res_dual.message}")
# Multiplicamos por -1 para recuperar el valor máximo
ganancia_max = -res_dual.fun
print(f"Ganancia maxima optima (Dual): {ganancia_max:.4f} centavos (esperado: 864.0)")
print(f"Precios sombra de las Vitaminas:")
print(f"  x1 (Vit A) = {res_dual.x[0]:.4f}")
print(f"  x2 (Vit B) = {res_dual.x[1]:.4f}")
print(f"  x3 (Vit C) = {res_dual.x[2]:.4f}")
print("-" * 50)

# 3. Verificación de la Dualidad Fuerte
print(f"¿Costo minimo del Primal == Ganancia maxima del Dual? {np.isclose(res_primal.fun, ganancia_max)}")



In [ ]:
# Graficación de la región factible no acotada del problema Primal (Nutrición)
# y1 + 2y2 >= 120 => y2 = (120 - y1)/2
# 2y1 + 3y2 >= 180 => y2 = (180 - 2y1)/3
# 3y1 + y2 >= 150 => y2 = 150 - 3y1

y1_vals = np.linspace(0, 140, 400)
y2_a = (120 - y1_vals) / 2.0
y2_b = (180 - 2 * y1_vals) / 3.0
y2_c = 150 - 3 * y1_vals

plt.figure(figsize=(10, 8))

# Graficamos las rectas de frontera
plt.plot(y1_vals, y2_a, 'r-', label='Vit A: $y_1 + 2y_2 \\ge 120$')
plt.plot(y1_vals, y2_b, 'g-', label='Vit B: $2y_1 + 3y_2 \\ge 180$')
plt.plot(y1_vals, y2_c, 'b-', label='Vit C: $3y_1 + y_2 \\ge 150$')

# Región factible: es la región por ENCIMA de todas las rectas de restricción
y_factible_nutr = np.maximum(y2_a, y2_b)
y_factible_nutr = np.maximum(y_factible_nutr, y2_c)
y_factible_nutr = np.maximum(y_factible_nutr, 0)

# Rellenamos hasta un límite arbitrario superior (p.ej. 160)
plt.fill_between(y1_vals, y_factible_nutr, 160, color='lightblue', alpha=0.3, label='Region Factible (No Acotada)')

# Vértices correctos
vertices_n = [(0, 150), (36, 42), (120, 0)]
costos_n = [10*v[0] + 12*v[1] for v in vertices_n]

for v, cost in zip(vertices_n, costos_n):
    color = 'gold' if v == (36, 42) else 'black'
    plt.scatter(v[0], v[1], color=color, s=120, zorder=5)
    plt.annotate(f"Vertice {v}\nCosto: {cost} centavos", xy=v, xytext=(v[0]+3, v[1]+3),
                 fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5))

plt.title('Metodo Grafico - Minimizacion de Costo de Nutricion (Region No Acotada)', fontsize=13, fontweight='bold')
plt.xlabel('Alimento 1: y1 (g)', fontsize=11)
plt.ylabel('Alimento 2: y2 (g)', fontsize=11)
plt.xlim(-5, 140)
plt.ylim(-5, 160)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(loc='upper right')
plt.show()



## 5. Resumen de la Lección

1. **Programación Lineal:** Es la técnica de planificación matemática para maximizar beneficios o minimizar costos sujeta a relaciones lineales.
2. **Método Gráfico:** Permite resolver modelos con dos variables de decisión graficando la región factible e identificando el punto esquina óptimo.
3. **Forma Estándar y de Holgura:** La conversión de inecuaciones a ecuaciones es el paso requerido para aplicar el algoritmo Simplex, introduciendo variables de holgura ($s$) o de exceso.
4. **Teoría de la Dualidad:** Cada problema primal tiene un problema dual equivalente. La resolución de cualquiera de ellos entrega información del otro.
5. **Teorema Fuerte de la Dualidad:** Los valores óptimos del primal y del dual coinciden exactamente ($c^T x^* = b^T y^*$).
6. **Resolución Computacional:** Librerías como `scipy.optimize.linprog` automatizan la búsqueda del óptimo utilizando métodos avanzados como el Símplex o Métodos de Punto Interior.

---

## 6. Referencias Bibliográficas

1. Boirivant, J. (2009). *La programación lineal aplicación de la pequeñas y medianas empresas*. Revista de Ciencias Económicas.
2. *Programación lineal*. Wikipedia, La enciclopedia libre.
3. *Linear programming*. Wikipedia, The Free Encyclopedia.
4. O’Connor, J. J. (1997). *Técnicas de cálculo para sistemas de ecuaciones, programación lineal y programación entera*. Madrid: Reverte.
5. del Valle Sotelo, J. C. (2012). *Álgebra lineal para estudiantes de ingeniería y ciencias*. Mc Graw Hill.
6. *Guía Interactiva con ejemplo de programación lineal*. Ejercicios y Ecuaciones.
